# conv-padding-zero — faded example 3: Asymmetric 2-D padding — index the interior window

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-padding-zero`. Running the beacon reports progress on the `CNN: Conv zero padding` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: Conv zero padding` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`conv-padding-zero`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "conv-padding-zero"
DD_SUBTOPIC = "CNN: Conv zero padding"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

With four independent side amounts (`top, bottom, left, right`), the padded shape is `(B, IC, top+H+bottom, left+W+right)` and the original belongs in the row window `[top : top+H]` and column window `[left : left+W]`. The single interior slice assignment is the load-bearing step.

## Faded exercise 3

Implement `pad2d_asym(x, top, bottom, left, right)` for `x` of shape `(B, IC, H, W)`. The zero buffer is allocated for you — complete the slice assignment that writes `x` into the correct interior row/column window so the four borders stay zero.

**Fill in:** the slice assignment writing x into rows [top:top+H] and columns [left:left+W] of out

In [ ]:
def pad2d_asym(x: Tensor, top: int, bottom: int, left: int, right: int) -> Tensor:
    B, IC, H, W = x.shape
    out = x.new_zeros(B, IC, top + H + bottom, left + W + right)
    raise NotImplementedError()  # TODO: the slice assignment writing x into rows [top:top+H] and columns [left:left+W] of out
    return out


def _test():
    t.manual_seed(0)
    cases = [(0, 0, 0, 0), (2, 1, 0, 3), (1, 0, 2, 0), (3, 2, 1, 4)]
    for top, bottom, left, right in cases:
        x = t.randn(2, 3, 5, 4)
        y = pad2d_asym(x, top, bottom, left, right)
        H, W = x.shape[2], x.shape[3]
        assert tuple(y.shape) == (2, 3, top + H + bottom, left + W + right)
        ref = t.nn.functional.pad(x, (left, right, top, bottom))
        assert t.allclose(y, ref)
        assert t.allclose(y[..., top:top + H, left:left + W], x)
        if top > 0:
            assert (y[..., :top, :] == 0).all()
        if bottom > 0:
            assert (y[..., -bottom:, :] == 0).all()
        if left > 0:
            assert (y[..., :, :left] == 0).all()
        if right > 0:
            assert (y[..., :, -right:] == 0).all()


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def pad2d_asym(x: Tensor, top: int, bottom: int, left: int, right: int) -> Tensor:
    B, IC, H, W = x.shape
    out = x.new_zeros(B, IC, top + H + bottom, left + W + right)
    out[..., top:top + H, left:left + W] = x
    return out
```
</details>